#Checking for Missing Files

In [ ]:
import os
import json
import re

def normalize_key(key):
    """
    Cleans the topic name key by aggressively removing all special characters,
    preserving only letters, numbers, and spaces. This is the most reliable
    method for matching against file names.
    """
    # 1. Replace all non-alphanumeric characters with a single space.
    # This removes colons, backslashes, commas, underscores, hyphens, and any other symbols.
    key = re.sub(r'[^a-zA-Z0-9\s]', ' ', key)

    # 2. Final cleanup: lowercase and consolidate multiple spaces
    key = key.lower()

    # Consolidate multiple spaces into one and strip leading/trailing spaces
    return ' '.join(key.split()).strip()

def validate_file_mapping(mapping_file_path, articles_folder_path):
    """
    Reads a mapping file and checks all article JSON files in a folder
    to see if a corresponding *aggressively normalized* key exists in the map.
    It reports on unmatched files only.
    """

    unmatched_files = []

    # 1. Read and process the mapping file
    print(f"📖 Reading mapping file: **{mapping_file_path}**...")
    try:
        with open(mapping_file_path, 'r', encoding='utf-8') as f:
            mapping_data = json.load(f)
    except Exception as e:
        print(f"❌ Error reading mapping file: {e}")
        return

    # Build the set of *aggressively normalized* keys for quick lookup
    taxonomy_keys = set()
    for item in mapping_data:
        openalex_name = item.get("openalex_name")
        if openalex_name:
            # Normalize the key from the JSON file
            normalized_key = normalize_key(openalex_name)
            taxonomy_keys.add(normalized_key)

    print(f"✅ Loaded {len(taxonomy_keys)} unique, normalized topics from the mapping file.")

    # Define the prefix to remove for robustness
    COPY_PREFIX = "copy of "

    # 2. Iterate through article files and validate
    print(f"\n📂 Validating files in folder: **{articles_folder_path}**...")
    total_files_checked = 0

    for filename in os.listdir(articles_folder_path):
        # Only check files that end with .json and are not the mapping file itself
        if filename.endswith(".json") and filename != os.path.basename(mapping_file_path):
            total_files_checked += 1

            # --- Key Generation for File ---
            # 1. Get base key
            file_key_for_lookup = filename.replace(".json", "")

            # 2. Remove the "copy of" prefix if present (case-insensitive check)
            if file_key_for_lookup.lower().startswith(COPY_PREFIX):
                file_key_for_lookup = file_key_for_lookup[len(COPY_PREFIX):].strip()

            # 3. Apply the same aggressive normalization logic as the JSON keys
            file_key_for_lookup = normalize_key(file_key_for_lookup)
            # --------------------------------

            # Check for a match
            if file_key_for_lookup in taxonomy_keys:
                pass
            else:
                print(f"  ❌ **UNMATCHED**: **{filename}** (Looked for key: '{file_key_for_lookup}')")
                unmatched_files.append(filename)


    # 3. Final Summary Report
    print("\n" + "="*50)
    print("✨ **MAPPING VALIDATION SUMMARY** ✨")
    print("=" * 50)
    print(f"🔍 Total Files Checked: **{total_files_checked}**")
    print(f"❌ Files Without a Match: **{len(unmatched_files)}**")

    if unmatched_files:
        print("\n**List of Files MISSING a Match in the Taxonomy:**")
        for i, filename in enumerate(unmatched_files, 1):
            print(f"  {i}. {filename}")

    print("=" * 50)


# --- Configuration ---
# ⚠️ UPDATE THESE PATHS ⚠️
MAPPING_FILE = '/content/merged_cleaned_chunks.json'
ARTICLES_INPUT_FOLDER = '/content/drive/MyDrive/NLP Data/Rayane_Preprocessing'

# --- Execution ---
if __name__ == "__main__":
    validate_file_mapping(
        mapping_file_path=MAPPING_FILE,
        articles_folder_path=ARTICLES_INPUT_FOLDER
    )

#Annotation

In [ ]:
import os
import json
import re

def clean_taxonomy_path(path_with_number):
    """
    Removes the numerical code (like '(6.04)') from the taxonomy path string.
    """
    cleaned_path = re.sub(r'\s*\(\d+\.\d+\)\s*', ' ', path_with_number).strip()
    cleaned_path = cleaned_path.replace(' >  > ', ' > ').replace(' > > ', ' > ')
    return cleaned_path

def normalize_key(key):
    """
    Cleans the topic name key by aggressively removing all special characters,
    preserving only letters, numbers, and spaces for robust matching.
    """
    key = re.sub(r'[^a-zA-Z0-9\s]', ' ', key)
    key = key.lower()
    return ' '.join(key.split()).strip()

def process_articles_and_classify(mapping_file_path, articles_folder_path):
    """
    Reads the mapping, classifies articles in the folder, and SAVES THE MODIFIED
    DATA BACK TO THE ORIGINAL FILE (overwriting it).

    Args:
        mapping_file_path (str): The path to the JSON file containing the mapping.
        articles_folder_path (str): The path to the folder containing the
                                    OpenAlex article JSON files.
    """

    # 1. Read and process the mapping file
    print(f"📖 Reading mapping file: **{mapping_file_path}**...")
    try:
        with open(mapping_file_path, 'r', encoding='utf-8') as f:
            mapping_data = json.load(f)
    except Exception as e:
        print(f"❌ Error reading mapping file: {e}")
        return

    # Create a dictionary for quick lookup: normalized key -> cleaned path
    taxonomy_map = {}
    for item in mapping_data:
        openalex_name = item.get("openalex_name")
        taxonomy_path = item.get("your_taxonomy_path")

        if openalex_name and taxonomy_path:
            # Normalize the JSON key before storage
            normalized_key = normalize_key(openalex_name)
            cleaned_path = clean_taxonomy_path(taxonomy_path)

            taxonomy_map[normalized_key] = cleaned_path

    # Define the prefix to remove for robustness
    COPY_PREFIX = "copy of "

    # 2. Iterate through article files and apply classification
    print(f"\n📂 Processing and saving articles back to folder: **{articles_folder_path}**...")
    processed_count = 0

    for filename in os.listdir(articles_folder_path):
        if filename.endswith(".json") and filename != os.path.basename(mapping_file_path):

            # --- Key Generation for File ---
            file_key_for_lookup = filename.replace(".json", "")

            if file_key_for_lookup.lower().startswith(COPY_PREFIX):
                file_key_for_lookup = file_key_for_lookup[len(COPY_PREFIX):].strip()

            # Apply the aggressive normalization to the filename key
            file_key_for_lookup = normalize_key(file_key_for_lookup)
            # --------------------------------

            full_file_path = os.path.join(articles_folder_path, filename)

            # Check for a match
            if file_key_for_lookup in taxonomy_map:
                classification_path = taxonomy_map[file_key_for_lookup]

                print(f"  -> Classifying **{filename}** (Key: '{file_key_for_lookup}') with path: **'{classification_path}'**")

                try:
                    # Read the articles file
                    with open(full_file_path, 'r', encoding='utf-8') as f:
                        articles_list = json.load(f)

                    # Update the 'classification_path'
                    for article in articles_list:
                        if isinstance(article, dict):
                            article["classification_path"] = classification_path

                    # CRITICAL: Write the modified list back to the SAME file path
                    with open(full_file_path, 'w', encoding='utf-8') as f:
                        json.dump(articles_list, f, indent=4, ensure_ascii=False)

                    processed_count += 1

                except Exception as e:
                    print(f"  ❌ Error processing file {filename}: {e}")
            else:
                print(f"  ⚠️ Skipping **{filename}**: No mapping found for normalized key '{file_key_for_lookup}'")

    print(f"\n🎉 Classification complete! **{processed_count}** files were classified and updated in place.")

# --- Configuration ---
# ⚠️ UPDATE THESE PATHS ⚠️
MAPPING_FILE = '/content/merged_cleaned_chunks.json'
# NOTE: The ARTICLES_INPUT_FOLDER is also the output folder now.
ARTICLES_INPUT_FOLDER = '/content/drive/MyDrive/NLP Data/Rayane_Preprocessing/Annotation_Test'

# --- Execution ---
if __name__ == "__main__":
    # Removed output_folder_path argument since we are overwriting the input files
    process_articles_and_classify(
        mapping_file_path=MAPPING_FILE,
        articles_folder_path=ARTICLES_INPUT_FOLDER
    )